# Load dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("nam194/vietnews")

In [ ]:
ds

In [ ]:
train_dataset = ds['train']
valid_dataset = ds['validation']
test_dataset = ds['test']

In [ ]:
import pandas as pd 

train_df = pd.DataFrame(train_dataset)
test_df = pd.DataFrame(test_dataset)
val_df = pd.DataFrame(valid_dataset)

In [ ]:
ds['train'][0]

# PreProcessing, tokenizer, build vocab

In [ ]:
import re
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

class Preprocessor:
    def __init__(self, max_len_content=300, max_len_summary=30, num_words_content=None, num_words_summary=None):
        self.max_len_content = max_len_content
        self.max_len_summary = max_len_summary
        self.num_words_content = num_words_content
        self.num_words_summary = num_words_summary
        
        self.content_tokenizer = None
        self.summary_tokenizer = None

    def clean_text(self, text, keep_punct=True):
        text = text.lower()
        #text = re.sub(r"[^a-zA-Z0-9áàảãạăắằẳẵặâấầẩẫậóòỏõọôốồổỗộơớờởỡợéèẻẽẹêếềểễệíìỉĩịúùủũụưứừửữựýỳỷỹỵđ.,!?]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def fit_tokenizers(self, contents, summaries):
        self.content_tokenizer = Tokenizer(num_words=self.num_words_content, oov_token="<unk>")
        self.content_tokenizer.fit_on_texts(contents)
        
        self.summary_tokenizer = Tokenizer(num_words=self.num_words_summary, oov_token="<unk>")
        self.summary_tokenizer.fit_on_texts(summaries)

    def transform(self, contents, summaries):
        # Clean
        contents = [self.clean_text(t) for t in contents]
        summaries = [self.clean_text(t) for t in summaries]
        
        # Thêm token đặc biệt cho summary
        summaries = ["<sos> " + s + " <eos>" for s in summaries]
        
        # Tokenize
        content_seq = self.content_tokenizer.texts_to_sequences(contents)
        summary_seq = self.summary_tokenizer.texts_to_sequences(summaries)
        
        # Padding
        content_pad = pad_sequences(content_seq, maxlen=self.max_len_content, padding='post')
        summary_pad = pad_sequences(summary_seq, maxlen=self.max_len_summary, padding='post')
        
        return content_pad, summary_pad

    def get_vocab_size(self):
        content_vocab_size = len(self.content_tokenizer.word_index) + 1
        summary_vocab_size = len(self.summary_tokenizer.word_index) + 1
        return content_vocab_size, summary_vocab_size


In [ ]:
contents = train_df["article"].astype(str).tolist()
summaries = train_df["abstract"].astype(str).tolist()

pre = Preprocessor(max_len_content=100, max_len_summary=15)

pre.fit_tokenizers(contents, summaries)

X_train, y_train = pre.transform(train_df['article'], train_df['abstract'])
X_val, y_val = pre.transform(val_df['article'], val_df['abstract'])
X_test, y_test = pre.transform(test_df['article'], test_df['abstract'])

content_vocab, summary_vocab = pre.get_vocab_size()

In [ ]:
print("X shape:", X_train.shape)
print("y shape:", y_train.shape)
print("Content vocab size:", content_vocab)
print("Summary vocab size:", summary_vocab)                                                            

In [ ]:
text = ["Hôm nay là ngày 18/9 , tôi đang cảm thấy dồi dào năng lượng . Do đó tôi quyết định sẽ học và làm một số bài tập về NLP ."]
summary = ["Tôi cảm thấy dồi dào năng lượng , tôi quyết định học và làm bài tập NLP . "]

text_sequence, summary_sequence = pre.transform(text, summary)

print(text_sequence)
print(summary_sequence)

In [ ]:
text = [ds['train'][0]['article']]
summary = [ds['train'][0]['abstract']]

text_sequence, summary_sequence = pre.transform(text, summary)

print(text_sequence)
print(summary_sequence)

In [ ]:
import numpy as np

decoded_text = pre.content_tokenizer.sequences_to_texts(
    [seq[seq != 0].tolist() for seq in np.array(text_sequence)]
)
decoded_summary = pre.summary_tokenizer.sequences_to_texts(
    [seq[seq != 0].tolist() for seq in np.array(summary_sequence)]
)

print("Decoded text:", decoded_text)
print("Decoded summary:", decoded_summary)

In [ ]:
print(list(pre.content_tokenizer.word_index.keys())[:1000])

In [ ]:
content_vocab_size, summary_vocab_size = pre.get_vocab_size()
INPUT_DIM  = content_vocab_size   
OUTPUT_DIM = summary_vocab_size

print(INPUT_DIM)
print(OUTPUT_DIM)

# Dataloader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextSummaryDataset(Dataset):
    def __init__(self, contents, summaries):
        self.contents = contents
        self.summaries = summaries

    def __len__(self):
        return len(self.contents)

    def __getitem__(self, idx):
        content = torch.tensor(self.contents[idx], dtype=torch.long)
        summary = torch.tensor(self.summaries[idx], dtype=torch.long)
        return content, summary

In [ ]:
train_dataset = TextSummaryDataset(X_train, y_train)
val_dataset   = TextSummaryDataset(X_val, y_val)
test_dataset  = TextSummaryDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Build Model

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.rnn = nn.GRU(embed_size, hidden_size, num_layers=num_layers,
                          batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, hidden_size)  # combine directions

    def forward(self, x):
        emb = self.embedding(x)  # (N, src_len, embed_size)
        outputs, hidden = self.rnn(emb)  # outputs: (N, src_len, 2*hidden)
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        return outputs, hidden.unsqueeze(0)

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W1 = nn.Linear(hidden_size * 2, hidden_size)
        self.W2 = nn.Linear(hidden_size, hidden_size)
        self.V = nn.Linear(hidden_size, 1)

    def forward(self, encoder_outputs, hidden):
        # encoder_outputs: (N, src_len, 2*hidden)
        # hidden: (1, N, hidden)
        hidden = hidden.permute(1, 0, 2)  # (N, 1, hidden)
        score = self.V(torch.tanh(self.W1(encoder_outputs) + self.W2(hidden)))
        attn_weights = torch.softmax(score, dim=1)  # (N, src_len, 1)
        context = torch.sum(attn_weights * encoder_outputs, dim=1)  # (N, 2*hidden)
        return context, attn_weights

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.rnn = nn.GRU(embed_size + hidden_size*2, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size*3, vocab_size)
        self.attention = BahdanauAttention(hidden_size)

    def forward(self, x, hidden, encoder_outputs):
        # x: (N) current token ids
        x = x.unsqueeze(1)  # (N,1)
        emb = self.embedding(x)  # (N,1,embed)
        context, _ = self.attention(encoder_outputs, hidden)  # (N, 2*hidden)
        context = context.unsqueeze(1)  # (N,1,2*hidden)
        rnn_input = torch.cat((emb, context), dim=-1)  # (N,1,embed+2*hidden)

        output, hidden = self.rnn(rnn_input, hidden)  # (N,1,hidden)
        output = torch.cat((output, context), dim=-1)  # (N,1,hidden+2*hidden)
        output = self.fc(output.squeeze(1))  # (N,vocab_size)
        return output, hidden

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.size()
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, trg_len, vocab_size).to(self.device)

        encoder_outputs, hidden = self.encoder(src)

        input_token = trg[:, 0]  # token đầu tiên <sos>

        for t in range(1, trg_len):
            output, hidden = self.decoder(input_token, hidden, encoder_outputs)
            outputs[:, t, :] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_token = trg[:, t] if teacher_force else top1

        return outputs


# Train

In [ ]:
from tqdm import tqdm

def train(model, iterator, optimizer, criterion, clip=1):
    model.train()
    epoch_loss = 0

    for src, trg in tqdm(iterator, desc="Training", leave=False):
        src, trg = src.to(device), trg.to(device)

        optimizer.zero_grad()
        output = model(src, trg)  # (N, trg_len, vocab_size)

        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)   # bỏ <sos>
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)


def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, trg in tqdm(iterator, desc="Evaluating", leave=False):
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, teacher_forcing_ratio=0)  

            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EMBED_SIZE = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 1

encoder = Encoder(INPUT_DIM, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS)
decoder = Decoder(OUTPUT_DIM, EMBED_SIZE, HIDDEN_SIZE)
model = Seq2Seq(encoder, decoder, device).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)  # 0 = <pad>
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
import time

N_EPOCHS = 5
for epoch in range(N_EPOCHS):
    start_time = time.time()

    train_loss = train(model, train_loader, optimizer, criterion)
    valid_loss = evaluate(model, val_loader, criterion)

    end_time = time.time()
    epoch_mins, epoch_secs = divmod(int(end_time - start_time), 60)

    print(f"Epoch {epoch+1}/{N_EPOCHS} | Time: {epoch_mins}m {epoch_secs}s")
    print(f"\tTrain Loss: {train_loss:.3f}")
    print(f"\t Val. Loss: {valid_loss:.3f}")